In [1]:
import numpy as np

import pandas as pd
pd.options.plotting.backend = "plotly"

import os
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
import scipy
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.filterwarnings('once')

In [2]:
import tensorflow as tf
tf.__version__

2025-12-22 21:00:21.918236: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-22 21:00:21.949114: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-22 21:00:23.582804: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


'2.19.0'

# Processamento e seleção da melhor coluna

In [3]:
best_column = 'NDCG@15'
metric_type = ["Prec", "Rec", "F1_Score", "Hit_Rate", "NDCG"]
top_k = [3, 5, 10, 20]

main_path = "results/metrics/validation"
main_file = os.listdir(main_path)

dataframes = []

for dataset_name in main_file:
        
    recommender_files = os.listdir(os.path.join(main_path, dataset_name))
    for recommender_name in recommender_files:
                
        metrics_path = os.path.join(main_path, dataset_name, recommender_name, "metrics.csv")
        metrics_aux = pd.read_csv(metrics_path, sep=';')
        
        metrics_aux.insert(0,'Recommender','')
        metrics_aux['Recommender'] = recommender_name

        metrics_aux.insert(0,'Dataset','')
        metrics_aux['Dataset'] = dataset_name
        
        dataframes.append(metrics_aux)
                
metrics_df = pd.concat(dataframes, ignore_index=True)

if (best_column == 'Mean'):
    metrics_df.insert(3,'Mean', metrics_df.mean(axis=1))
    
if best_column in metrics_df.columns:
    saida = metrics_df.loc[metrics_df.reset_index().groupby(['Dataset', 'Recommender'])[best_column].idxmax()].reset_index(drop=True)
else:
    raise Exception("Coluna '" + best_column + "' não encontrada")

# Análise de resultados

## 1- Geral

In [4]:
pd.options.display.max_colwidth = 500
df_display = saida[saida.Recommender != "ALS"][["Dataset", "Recommender", "Parameters", "NDCG@20"]]
df_display = df_display[saida.Recommender != "BPR"]
display(saida)

/tmp/ipykernel_1065725/27259134.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_display = df_display[saida.Recommender != "BPR"]


,Dataset,Recommender,Parameters,Prec@1,Prec@2,Prec@3,Prec@4,Prec@5,Prec@6,Prec@7,...,NDCG@11,NDCG@12,NDCG@13,NDCG@14,NDCG@15,NDCG@16,NDCG@17,NDCG@18,NDCG@19,NDCG@20
0,amazon-beauty,ALS,factors=100@iterations=10@regularization=1e-06,0.004444,0.003333,0.004444,0.003333,0.002889,0.002778,0.002540,...,0.011294,0.011568,0.011833,0.011833,0.012086,0.012086,0.012571,0.012571,0.012805,0.013035
1,amazon-beauty,BPR,factors=50@iterations=100@learning_rate=0.25@regularization=0.01,0.012222,0.010556,0.008148,0.006389,0.005333,0.005370,0.004762,...,0.022179,0.022179,0.022444,0.022444,0.022444,0.022444,0.022444,0.022444,0.022444,0.022444
2,amazon-beauty,Item2Vec_itemSim,combination_strategy=avg_norm_before@epochs=20@learning_rate=0.025@negative_exp=0.5@negative_samples=5@recomender_norm=True@regularization=-1.0@subsample=0.001@w_size=10,0.014444,0.010000,0.008148,0.006389,0.005778,0.005185,0.004444,...,0.024017,0.024291,0.024556,0.024815,0.025068,0.025563,0.025805,0.025805,0.025805,0.025805
3,amazon-beauty,TimeI2V_Cont,combination_strategy=avg_norm_before@decay_rate=-1.0@epochs=80@learning_rate=0.025@min_time_diff=300@negative_exp=1@negative_samples=5@recomender_norm=True@regularization=-1.0@subsample=0.001@w_size=10@weight_floor=0.3,0.015556,0.010000,0.006667,0.006944,0.006222,0.005370,0.004921,...,0.024042,0.025135,0.025667,0.026184,0.026437,0.026932,0.026932,0.026932,0.026932,0.026932
4,amazon-beauty,TimeI2V_Disc_Aug,combination_strategy=avg_norm_before@epochs=20@learning_rate=0.025@min_time_diff=300@negative_exp=0.5@negative_samples=5@recomender_norm=True@regularization=-1.0@subsample=0.001@time_exp=1.5@w_size=10,0.012222,0.008333,0.006667,0.005556,0.004667,0.004815,0.005079,...,0.022916,0.023463,0.023463,0.023980,0.024486,0.024733,0.024733,0.024972,0.025439,0.025439
5,ciaodvd,ALS,factors=200@iterations=100@regularization=1e-06,0.028986,0.021739,0.016103,0.013285,0.012560,0.010467,0.008972,...,0.019261,0.020201,0.021113,0.022448,0.022882,0.022882,0.023299,0.024936,0.025740,0.025740
6,ciaodvd,BPR,factors=200@iterations=50@learning_rate=0.025@regularization=0.01,0.019324,0.019324,0.017713,0.014493,0.013527,0.012077,0.011732,...,0.020869,0.021339,0.021339,0.021339,0.021773,0.022198,0.022198,0.022607,0.022607,0.023003
7,ciaodvd,Item2Vec_itemSim,combination_strategy=avg_norm_before@epochs=80@learning_rate=0.25@negative_exp=-0.5@negative_samples=5@recomender_norm=True@regularization=-1.0@subsample=0.001@w_size=10,0.009662,0.009662,0.008052,0.007246,0.007729,0.008052,0.008282,...,0.013656,0.014595,0.015051,0.015941,0.016810,0.017235,0.017652,0.017652,0.018858,0.019254
8,ciaodvd,TimeI2V_Cont,combination_strategy=avg_norm_before@decay_rate=-1.0@epochs=20@learning_rate=0.025@min_time_diff=300@negative_exp=-0.5@negative_samples=5@recomender_norm=True@regularization=-1.0@subsample=0.001@w_size=5@weight_floor=0.3,0.024155,0.016908,0.012882,0.010870,0.008696,0.007246,0.006901,...,0.014661,0.016540,0.016996,0.017886,0.018320,0.018320,0.019154,0.020381,0.021185,0.021977
9,ciaodvd,TimeI2V_Disc_Aug,combination_strategy=avg_norm_before@epochs=20@learning_rate=0.025@min_time_diff=300@negative_exp=-0.5@negative_samples=5@recomender_norm=True@regularization=-1.0@subsample=0.001@time_exp=1.5@w_size=5,0.019324,0.014493,0.012882,0.010870,0.009662,0.008857,0.007591,...,0.015462,0.015462,0.016375,0.016820,0.016820,0.016820,0.017236,0.017645,0.018450,0.018450


## 2 - Por parâmetros

In [ ]:
pd.set_option('display.max_rows', 300)

def metrics_per_parameter(df, df_name, parameters):

    def extract_parameter_value(param_str, target_param):
        params = dict(item.split('=') for item in param_str.split('@'))
        return params.get(target_param, None)

    if df_name != 'all':
        df = df[df['Dataset'] == df_name].copy()
        
    df = df[df['Recommender'] != 'ALS']
    df = df[df['Recommender'] != 'BPR']
       
    for parameter_name in parameters:
        df[parameter_name] = df['Parameters'].apply(lambda x: extract_parameter_value(x, parameter_name))
    
    best_values = (
        df.groupby(['Dataset', 'Recommender'] + parameters)['NDCG@20']
        .max()
        .reset_index()
        .sort_values(by=['Dataset', 'Recommender', 'NDCG@20'], ascending=[True, True, False])
    )

    return best_values

metrics_per_parameter(metrics_df, 'all', ['subsample'])

,Dataset,Recommender,learning_rate,NDCG@20
0,amazon-beauty,Item2Vec_itemSim,0.025,0.025805
1,amazon-beauty,Item2Vec_itemSim,0.25,0.024662
2,amazon-beauty,TimeI2V_Cont,0.025,0.027117
3,amazon-beauty,TimeI2V_Cont,0.25,0.025966
4,amazon-beauty,TimeI2V_Disc_Aug,0.025,0.026144
5,amazon-beauty,TimeI2V_Disc_Aug,0.25,0.024815
7,ciaodvd,Item2Vec_itemSim,0.25,0.019254
6,ciaodvd,Item2Vec_itemSim,0.025,0.016506
8,ciaodvd,TimeI2V_Cont,0.025,0.021977
9,ciaodvd,TimeI2V_Cont,0.25,0.019101


# Geração de gráficos

In [16]:
def function_1(x):
    if(len(x.split('@')) > 1): 
        x = x.split('@')[1]
    return x

curr_metric = "Hit_Rate"

main_path = "results/metrics"
main_file = os.listdir(main_path)

for dataset_name in main_file:
    
    saida_aux = saida[saida['Dataset'] == dataset_name] 
        
    my_regex = "Recommender|" +  curr_metric + ".*"
    df_aux = saida_aux.filter(regex=(my_regex))
    
    df_aux = df_aux.set_index('Recommender')
    df_aux = df_aux.rename(columns=function_1)
    
    marker_symbols = ['circle', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagram']

    fig = go.Figure()

    for i, recomendador in enumerate(df_aux.index):
        fig.add_trace(
            go.Scatter(
                x=df_aux.columns,  # Colunas de métricas
                y=df_aux.loc[recomendador],  # Pontuações do recomendador
                mode='lines+markers',
                marker=dict(symbol=marker_symbols[i], size=7, line=dict(color='black', width=0.5)),
                name=recomendador
            )
        )
    
    fig.update_layout(
    title=dataset_name+" - "+curr_metric,
    title_x=0.15,
    title_y=0.8,
    showlegend=True,
    height=400,
    width=600,
    yaxis_title="",
    xaxis_title="",
    legend_title="Recommenders",
    font=dict(
        family="Helvetica",
        size=12,
        color="black"
        )
    )

    fig.show()

In [7]:
'''def function_1(x):
    if len(x.split('@')) > 1: 
        x = x.split('@')[1]
    return x

curr_metric = "NDCG"

main_path = "results/metrics"
main_file = os.listdir(main_path)

# Determine the grid size
rows = 1
cols = 1
num_plots = len(main_file)

# Define colors for each recommender
color_map = {
    'ALS - UI': '#1f77b4',
    'ALS - II': '#ff7f0e',
    'ALS - WS': '#2ca02c',
    'BPR - UI': '#d62728',
    'BPR - II': '#9467bd',
    'BPR - WS': '#bcbd22',
}

# Create subplots with adjusted width and height
fig = make_subplots(
    rows=rows, 
    cols=cols, 
    subplot_titles=main_file,
    vertical_spacing=0.05,  # Decrease vertical space between plots
    horizontal_spacing=0.05  # Decrease horizontal space between plots
)

# Iterate over datasets and create subplots
for idx, dataset_name in enumerate(main_file):
    row = (idx // cols) + 1
    col = (idx % cols) + 1
    
    saida_aux = saida[saida['Dataset'] == dataset_name] 
    my_regex = "Recommender|" +  curr_metric + ".*"
    df_aux = saida_aux.filter(regex=(my_regex))
    
    df_aux["Recommender"] = df_aux["Recommender"].replace(["ALS", "ALS_itemSim", "ALS_weighted", "BPR", "BPR_itemSim", "BPR_weighted"], 
                                                  ["ALS - UI", "ALS - II", "ALS - WS", "BPR - UI", "BPR - II", "BPR - WS"])
        
    df_aux = df_aux.set_index('Recommender')
    df_aux = df_aux.rename(columns=function_1)
    
    marker_symbols = ['circle', 'square', 'diamond', 'cross', 'triangle-up', 'triangle-down', 'star', 'hexagram']

    for i, recomendador in enumerate(df_aux.index):
        fig.add_trace(
            go.Scatter(
                x=df_aux.columns,  # Colunas de métricas
                y=df_aux.loc[recomendador],  # Pontuações do recomendador
                mode='lines+markers',
                marker=dict(symbol=marker_symbols[i], size=7, color=color_map.get(recomendador, 'black'), line=dict(color='black', width=0.5)),
                name=recomendador,
                showlegend=(idx == 6)  # Show legend only for the specified subplot
            ),
            row=row,
            col=col
        )

fig.update_layout(
    height=1400,  # Increase height
    width=800,   # Decrease width
    title_text="",  # Global title
    showlegend=True,
    font=dict(
        family="Helvetica",
        size=12,
        color="black"
    ),
    legend_title="Recommenders",
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.04,  
        xanchor='center',
        x=0.5,
        font=dict(
            size=12.5
        )
    )
)

fig.show()'''

'def function_1(x):\n    if len(x.split(\'@\')) > 1: \n        x = x.split(\'@\')[1]\n    return x\n\ncurr_metric = "NDCG"\n\nmain_path = "results/metrics"\nmain_file = os.listdir(main_path)\n\n# Determine the grid size\nrows = 1\ncols = 1\nnum_plots = len(main_file)\n\n# Define colors for each recommender\ncolor_map = {\n    \'ALS - UI\': \'#1f77b4\',\n    \'ALS - II\': \'#ff7f0e\',\n    \'ALS - WS\': \'#2ca02c\',\n    \'BPR - UI\': \'#d62728\',\n    \'BPR - II\': \'#9467bd\',\n    \'BPR - WS\': \'#bcbd22\',\n}\n\n# Create subplots with adjusted width and height\nfig = make_subplots(\n    rows=rows, \n    cols=cols, \n    subplot_titles=main_file,\n    vertical_spacing=0.05,  # Decrease vertical space between plots\n    horizontal_spacing=0.05  # Decrease horizontal space between plots\n)\n\n# Iterate over datasets and create subplots\nfor idx, dataset_name in enumerate(main_file):\n    row = (idx // cols) + 1\n    col = (idx % cols) + 1\n    \n    saida_aux = saida[saida[\'Dataset\']

# Geração de tabelas - Divisão por Top-N

In [8]:
# Gera uma tabela para todos os datasets em todos os Top-N diferentes
# Retorna o melhor parametro de cada dataset

def function_1(x):
   x = x.split('@')[0]
   return x

names_dict = {"Prec": "Precision", "Rec": "Recall", "F1_Score":"F1-Score", "Hit_Rate": "Hit-Rate"}

results_path = "latex_results"

string_final = ""

if not os.path.exists(results_path):
    os.makedirs(results_path)
    
main_file = os.listdir(main_path)

for dataset_name in main_file:
    
    saida_aux = saida[saida['Dataset'] == dataset_name] 
      
    string_final = string_final + ("=========== DATASET: " + dataset_name + " ===========\n\n\n")
    string_final = string_final + ("=========== MELHORES PARÂMETROS ===========\n\n")
    string_final = string_final + (saida[["Recommender", "Parameters"]].to_string())
    
    for curr_k in top_k:
        
        aux = '\n\n------ Top' + str(curr_k) + ' ------\n\n'
        string_final = string_final + (aux)

        #Pre processamento das colunas
        my_regex = "Recommender|" +  str(curr_k) + "$"
        df_aux = saida_aux.filter(regex=(my_regex))
        df_aux = df_aux.round(4)
        df_aux = df_aux.rename(columns=function_1)
        df_aux = df_aux.rename(columns=names_dict)

        # Tranforma a tabela em Latex
        capt = "Top " + str(curr_k) + " recommendation"
        string_final = string_final + (df_aux.style.hide(axis="index").set_caption(capt).to_latex())
        
        
    string_final = string_final + ("\n\n\n\n")

f = open(os.path.join(results_path, "saida.txt"), "w") 
f.write(string_final)
f.close()

# Geração de tabelas - Tabela unificada

In [9]:
def pre_proc(df, comp_metric):
    
    #Seleciona apenas a metrica relevante
    df_aux = df.loc[:, ["Dataset", "Recommender", comp_metric]].copy()

    #Separa a coluna de recomendadores em dois
    df_aux[['Embedding', 'Recommender']]= df_aux['Recommender'].str.split('_', expand=True)

    #Preenche os valores nulos
    df_aux['Recommender'] = np.where(df_aux['Recommender'].isna() , df_aux['Embedding'], df_aux['Recommender'])
    
    #Renomeia colunas
    df_aux["Recommender"] = df_aux["Recommender"].replace(names_dict)
    
    df_pivot = df_aux.pivot(index='Dataset', columns=['Embedding', 'Recommender'], values=comp_metric)
    df_pivot = df_pivot.reset_index()
    
    return df_pivot

In [10]:
def create_gray_scale(df_pivot):
    
    #Cria a escala de cinza
    GS = pd.concat([
        pd.DataFrame(MinMaxScaler(feature_range=(0.5, 0.95)).fit_transform(-df_pivot.iloc[:,1:4].values.T).round(4).T),
        pd.DataFrame(MinMaxScaler(feature_range=(0.5, 0.95)).fit_transform(-df_pivot.iloc[:,4:].values.T).round(4).T),
    ], axis=1)
    GS.index = df_pivot.index
    GS.columns = df_pivot.iloc[:,1:].columns
    
    return GS

In [11]:
def add_gain(df_optimal, df_real):
    
    real_metrics = df_real.values[:,1:]
    optimal_metrics = df_optimal.values[:,1:]
    
    gain_percentage = (((optimal_metrics - real_metrics)/real_metrics)*100).astype(float).round(1).astype(str)
    
    df_real.iloc[:,2:4] = df_real.round(4).astype(str).iloc[:,2:4] + " (+" +  gain_percentage[:,1:3] + "\%)"
    df_real.iloc[:,5:] = df_real.round(4).astype(str).iloc[:,5:] + " (+" +  gain_percentage[:,4:] + "\%)"
    
    return df_real

<>:8: DeprecationWarning:

invalid escape sequence '\%'

<>:8: DeprecationWarning:

invalid escape sequence '\%'



In [12]:
#comp_metric = "NDCG@10"

f = open(os.path.join(results_path, "saida_tabela_unificada.txt"), "w")

names_dict = {"itemSim": "Item-Item", "weighted": "Weighted", "ALS": "User-Item", "BPR": "User-Item"}

#Seleciona apenas as colunas terminadas em @10
metrics_names = saida.filter(regex='@10').columns

for comp_metric in metrics_names:
        
    df_final = pre_proc(saida, comp_metric)
    GS = create_gray_scale(df_final)
    
    #df_final = add_gain(df_optimal, df_real)
    
    #Coloca a escala na tabela
    df_final.iloc[:,1:] = "\\cellcolor[gray]{" + GS.astype(str) + "}" + df_final.round(4).astype(str).iloc[:,1:]
        
    tabela_latex = (df_final.style.hide(axis="index")
                    .to_latex(column_format='lccc|ccc', 
                              multicol_align='c',
                              hrules='True',
                              position_float='centering',
                              caption="Comparação de diversos métodos sob " + comp_metric))

    #Centraliza a tabela
    tabela_latex = tabela_latex.replace("\\begin{tabular}", "\\resizebox{\\textwidth} {!}{ \\begin{tabular}")
    tabela_latex = tabela_latex.replace("\\end{tabular}", "\\end{tabular} }")

    results_path = "latex_results"

    if not os.path.exists(results_path):
        os.makedirs(results_path)

    f.write("\n\n\n------------------  " + "MÉTRICA: " + comp_metric + " ------------------\n\n\n")
    f.write(tabela_latex)

f.close()

ValueError: Columns must be same length as key

ValueError: Columns must be same length as key

# Geração de tabelas - Comparação Item2Vec - TemporalI2V

In [ ]:
f = open(os.path.join(results_path, "saida_tabela_comparacao.txt"), "w")

comp_metric = "NDCG@10"
metrics_names = saida.filter(regex=comp_metric).columns

df_aux = saida.loc[:, ["Dataset", "Recommender", comp_metric]].copy()
df_aux = df_aux[df_aux['Recommender'] != 'ALS']
df_aux = df_aux[df_aux['Recommender'] != 'ALS_itemSim']
df_aux = df_aux[df_aux['Recommender'] != 'BPR']

print(df_aux)

df_pivot = df_aux.pivot(index='Dataset', columns='Recommender', values=comp_metric)
df_pivot = df_pivot.rename_axis(None, axis=1).reset_index()

for column in df_pivot.columns:
    if(column == 'Item2Vec_itemSim' or column == 'Dataset'):
        continue
    df_pivot[column] = round((df_pivot[column] - df_pivot['Item2Vec_itemSim'])/df_pivot['Item2Vec_itemSim'] * 100, 2)

df_pivot = df_pivot.drop(columns=['Item2Vec_itemSim'])
df_pivot.iloc[:,1:] = df_pivot.iloc[:,1:].astype(str) + '\%'

tabela_comparacao = df_pivot.style.hide(axis="index").to_latex()
tabela_comparacao = tabela_comparacao.replace("_", "-")

print(tabela_comparacao)

f.close()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

def data_generator(num_items_per_user=20):
    
    np.random.seed(0)

    # Define the quantities for each category
    num_zeros = int(num_items_per_user * 0.20)
    num_small = int(num_items_per_user * 0.70)
    num_big = int(num_items_per_user * 0.10)

    zeros = np.zeros(num_zeros, dtype=int)
    values_small = np.random.randint(10, 21, size=num_small)  # Random integers between 10 and 20
    values_big = np.random.randint(60, 200, size=num_big)  # Random integers between 90 and 100

    # Combine all values into a single array
    data = np.concatenate([zeros, values_small, values_big])
    
    while len(data) < num_items_per_user:
        data = np.concatenate([data, np.random.randint(10, 21, size=1)])

    # Shuffle the array
    np.random.shuffle(data)

    return data

# Function to plot interaction weights
def plot_interaction_weights(num_items_per_user, alpha, w_min, weight_proposal, time_diff_threshold, decay_threshhold):
    
    def calculate_proposal_1(s1, s2, scale):
        
        w = (1 - abs(s1 - s2))
        
        if (w + scale) > 1:
            return 1
        else:    
            return np.maximum((w+scale), 0.3)
            #return np.maximum(1 - (np.log10(1/(w+scale))), 0.3)
    
    # Define weight functions
    def calculate_proposal_2(z, alpha=alpha, w_min=w_min):
        if z < 0:
            return 1
        else:
            return 1 - np.power((z / (z + 1)), alpha)

    def calculate_proposal_3(z_scores, s1, norms, alpha=alpha, w_min=w_min):
        weight_1 = [calculate_proposal_1(s1, s2, scale) for s2 in norms]
        weight_2 = [calculate_proposal_2(z) for z in z_scores]
        return (np.array(weight_1) + np.array(weight_2))/2
        
    data = {
        'user_id': np.repeat([1], num_items_per_user),
        'item_id': list(range(1, num_items_per_user + 1)),
        'timestamp': data_generator(num_items_per_user)
    }
        
    df = pd.DataFrame(data)
          
    # Calculate cumulative time for each user
    df['cumulative_time'] = df.groupby('user_id')['timestamp'].cumsum()

    # Normalize cumulative time to [w_min, 1] using MinMaxScaler
    scaler = MinMaxScaler(feature_range=(w_min, 1))
    df['normalized_cumulative'] = df.groupby('user_id')['cumulative_time'].transform(
        lambda x: scaler.fit_transform(x.values.reshape(-1, 1)).flatten()
    )
        
    # Plot weights for each user
    for user_id in df['user_id'].unique():
        user_data = df[df['user_id'] == user_id].reset_index(drop=True)
                
        indices = {
            'Middle Item': len(user_data) // 2,
            'First Item': 0,
            'Last Item': len(user_data) - 1
        }

        fig = go.Figure()

        for label, idx in indices.items():
            
            reference_norm = user_data.loc[idx, 'normalized_cumulative']
            reference_cumsum = user_data.loc[idx, 'cumulative_time']
            
            non_noise_df = user_data[user_data['timestamp'] > time_diff_threshold]
            
            user_mean = non_noise_df['timestamp'].mean()
            user_std = non_noise_df['timestamp'].std()
            
            diff_cumsum = [abs(reference_cumsum - cumsum) for cumsum in user_data['cumulative_time']]
            
            z_scores = (diff_cumsum - user_mean) / user_std
            
            scale = scaler.scale_[0] * decay_threshhold
                        
            if weight_proposal == 1:
                weights = [calculate_proposal_1(reference_norm, norm, scale) for norm in user_data['normalized_cumulative']]
            elif weight_proposal == 2:
                weights = [calculate_proposal_2(z) for z in z_scores]
            elif weight_proposal == 3:
                weights = calculate_proposal_3(z_scores, reference_norm, user_data['normalized_cumulative'].values)
            

            # Plot the weight line
            fig.add_trace(go.Scatter(
                x=user_data['cumulative_time'],
                y=weights,
                mode='lines+markers',
                name=f'{label} (User {user_id})'
            ))

            # Add a vertical dotted line at the reference item's cumulative time
            fig.add_vline(
                x=user_data.loc[idx, 'cumulative_time'],
                line=dict(color="green", dash="dot"),
                annotation_text=label,
                annotation_position="top left"
            )

        # Update layout and display
        proposal_title = f"Proposal {weight_proposal}: "
        proposal_title += f"Mean: {round(user_mean, 2)}, STD {round(user_std, 2)}"
        fig.update_layout(
            title=f'{proposal_title} (User {user_id})',
            xaxis_title='Cumulative Time',
            yaxis_title='Weight',
            yaxis=dict(range=[0, 1.1]),
        )
        fig.show()

In [ ]:
num_items_per_user = 50
alpha = 2
w_min = 0
time_diff_threshold = 10
decay_threshhold = 200

# Fazer com que os valores decaiam apenas quando a distância entre dois itens for maior do que o treshhold
for weight_proposal in range(1, 4):
    plot_interaction_weights(num_items_per_user, alpha, w_min, weight_proposal, time_diff_threshold, decay_threshhold)

In [ ]:
df = pd.read_csv('datasets/NetflixPrize/Original/interactions.csv', sep=";", header='infer')
df.columns = ["id_user", "id_item", "rating", "datetime", "timestamp"]

In [ ]:
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
sampled_users = df['id_user'].drop_duplicates().sample(frac=0.01, random_state=42)
sampled_df = df[df['id_user'].isin(sampled_users)]

In [ ]:
sampled_df.to_csv('datasets/NetflixPrize/interactions.csv', sep=";")

In [ ]:
df = pd.read_csv('datasets/CiaoDVD/interactions_explicit.csv', sep=";", header='infer')
df.columns = ["id_user", "id_item", "rating", "datetime"]
df = df[df['rating'] > 3]
df = df.drop(columns = 'rating')

In [ ]:
df.to_csv('datasets/CiaoDVD/interactions.csv', sep=";", index=False)

In [ ]:
import numpy as np
import darts
from darts import TimeSeries
from darts.ad import KMeansScorer
import matplotlib.pyplot as plt

# Example data: Time intervals between interactions (in seconds)
time_intervals = [10, 12, 11, 500, 10, 13, 1000, 12, 11]

# Convert the time intervals into a TimeSeries object
series = TimeSeries.from_values(np.array(time_intervals))

# Initialize an anomaly scorer (e.g., KMeansScorer)
scorer = KMeansScorer(k=2)  # k=2 for two clusters (normal vs anomalous)

# Compute anomaly scores
anomaly_scores = scorer.score(series)

# Plot the time intervals and anomaly scores
plt.figure(figsize=(10, 6))
plt.plot(series.time_index, series.values(), label="Time Intervals")
plt.plot(series.time_index, anomaly_scores.values(), label="Anomaly Scores", color="red")
plt.axhline(y=np.mean(anomaly_scores.values()), color="orange", linestyle="--", label="Threshold")
plt.legend()
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Time Intervals and Anomaly Scores")
plt.show()

# Detect breakpoints (where anomaly scores are above a threshold)
threshold = np.mean(anomaly_scores.values())  # You can adjust this threshold
breakpoints = np.where(anomaly_scores.values() > threshold)[0]

# Split the interaction list at breakpoints
split_intervals = np.split(time_intervals, breakpoints)

# Print the results
print("Breakpoints detected at indices:", breakpoints)
print("Split interaction lists:")
for i, sublist in enumerate(split_intervals):
    print(f"List {i+1}: {sublist}")

In [ ]:
import tensorflow as tf

def add_ones_sequence(arr, num_ones):
    # Create a tensor of ones with shape (len(arr), num_ones)
    ones = tf.ones([tf.shape(arr)[0], num_ones], dtype=arr.dtype)
    
    # Stack the original array and ones along a new axis
    stacked = tf.concat([tf.expand_dims(arr, axis=1), ones], axis=1)
    
    # Reshape to a 1D tensor
    result = tf.reshape(stacked, [-1])
    return result

# Define a TensorFlow function for running inside a session
@tf.function
def run_tensorflow():
    # Input tensor
    arr = tf.constant([3, 5, 7], dtype=tf.float32)

    # Set hyperparameter: number of ones to insert
    num_ones = 3  # Change this value as needed

    # Apply transformation
    result = add_ones_sequence(arr, num_ones)

    return result

# Execute in TensorFlow eager mode
result_tensor = run_tensorflow()

# Print the result
print(result_tensor.numpy())  # Expected output: [3. 1. 1. 1. 5. 1. 1. 1. 7. 1. 1. 1.]

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

def separate_datetime_components(dt):
    if isinstance(dt, (int, float)):
        dt = datetime.fromtimestamp(dt)
    return np.array([dt.year, dt.month, dt.day, dt.hour])

# Example DataFrame with mixed datetime representations
df = pd.DataFrame({
    'dt': [datetime(2021, 1, 1, 12), datetime(2021, 1, 2, 13), 1609459200]  # last one is a Unix timestamp
})

# Apply the function to each element and stack the results into a 2D array
result = np.vstack(df['dt'].apply(separate_datetime_components))
print(result)


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime



def get_circle_partition_position(partition, N):

    angle = partition * (2 * np.pi / N)
    
    x = np.round(np.cos(angle), 3)
    y = np.round(np.sin(angle), 3)
    
    print(x)
    
    return x, y

In [ ]:
# Example usage:
N = 8  # Number of divisions in the circle
partition = 4  # The partition number (0 to N-1)
x, y = get_circle_partition_position(partition, N)
print(f"Partition {partition} is at position (x, y) = ({x:.2f}, {y:.2f})")

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime

class Date2Emb_model:
    def __init__(self, keys):
            
        self.CONSULTING_TABLE = {
            "Month": {"extractor": self._get_month, "unique_values": 12},
            "WeekDay": {"extractor": self._get_week_day, "unique_values": 7},
            "Hour": {"extractor": self._get_hour, "unique_values": 24},
            "Weekend": {"extractor": self._get_weekend, "unique_values": 2},
            "TimeOfDay": {"extractor": self._get_time_of_day, "unique_values": 3},
        }

        self.keys = keys
        self.partitions = self._get_unique_values()
        
    def _get_month(self, dt):
        return dt.month-1
    
    def _get_week_day(self, dt):
        return dt.weekday()-1
            
    def _get_hour(self, dt):
        return dt.hour-1

    def _get_weekend(self, dt):
        return 1 if dt.weekday() >= 5 else 0

    def _get_time_of_day(self, dt):
        return 0 if 5 <= dt.hour < 12 else 1 if 12 <= dt.hour < 18 else 2

    def _get_unique_values(self):
        return np.array([self.CONSULTING_TABLE[key]["unique_values"] for key in self.keys])

    def _get_extracted_values(self, dt):
        return np.array([self.CONSULTING_TABLE[key]["extractor"](dt) for key in self.keys])
    
    def _get_position(self, partition):

        angle = partition * (2 * np.pi / self.partitions)
        
        x = np.round(np.cos(angle), 3)
        y = np.round(np.sin(angle), 3)
        
        return np.stack((x, y), axis=-1).reshape(angle.shape[0], -1)

    def get_embeddings(self, df):
        
        if 'timestamp' in df.columns:
            df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
        elif 'datetime' in df.columns:
            df['datetime'] = pd.to_datetime(df['datetime']).dt.floor('s')

        extracted_time_values = np.vstack(df['datetime'].apply(self._get_extracted_values))        
        return self._get_position(extracted_time_values)

# Example DataFrame with mixed datetime representations
df = pd.DataFrame({
    'user_id':  [  1,   1,   1],
    'item_id':  [101, 102, 103],
    'datetime': [datetime(2021, 5, 10, 2), datetime(2021, 3, 11, 15), datetime(2021, 4, 12, 22)]
})

keys = ["Month", "Hour", "Weekend", "TimeOfDay"]
date2emb = Date2Emb_model(keys)
date_embeddings = date2emb.get_embeddings(df)
print(date_embeddings)

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime

class Date2Emb_model:
    def __init__(self, keys):

        self.CONSULTING_TABLE = pd.DataFrame([
            ["Month", self._get_month, 12],
            ["WeekDay", self._get_week_day, 7],
            ["Hour", self._get_hour, 24],
            ["Weekend", self._get_weekend, 2],
            ["TimeOfDay", self._get_time_of_day, 3],],
            columns=['Name', 'Extractor', 'Separations'])

        self.keys = keys
        self.partitions = self._get_unique_values()

    def _get_month(self, dt):
        return dt.month-1
    
    def _get_week_day(self, dt):
        return df.dt.weekday()-1
            
    def _get_hour(self, dt):
        return dt.hour-1

    def _get_weekend(self, dt):
        return 1 if dt.weekday() >= 5 else 0

    def _get_time_of_day(self, dt):
        return 0 if 5 <= dt.hour < 12 else 1 if 12 <= dt.hour < 18 else 2

    def _get_unique_values(self):
        filtered_table = self.CONSULTING_TABLE[self.CONSULTING_TABLE['Name'].isin(self.keys)]
        return filtered_table['Separations'].values

    def _get_extracted_values(self, dt):
        filtered_table = self.CONSULTING_TABLE[self.CONSULTING_TABLE['Name'].isin(self.keys)]
        extracted_values = filtered_table['Extractor'].apply(lambda func: func(dt))
        print(extracted_values)
        return extracted_values

    def _get_position(self, partition):
        angle = partition * (2 * np.pi / self.partitions)
        x = np.round(np.cos(angle), 3)
        y = np.round(np.sin(angle), 3)
        return np.column_stack((x, y))

    def get_embeddings(self, df):
        if 'timestamp' in df.columns:
            df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
        elif 'datetime' in df.columns:
            df['datetime'] = pd.to_datetime(df['datetime']).dt.floor('s')

        # Apply _get_extracted_values to all rows in the DataFrame
        extracted_time_values = np.vstack(df['datetime'].apply(self._get_extracted_values))
        return self._get_position(extracted_time_values)
    
keys = [ "Hour", "Month", "Weekend", "TimeOfDay"]
date2emb = Date2Emb_model(keys)
date_embeddings = date2emb.get_embeddings(df)
print(date_embeddings)